# Progetto di analisi del testo

## Domande di ricerca
- Parole ricercate - linguaggio alto/basso (misurata per regione, istruzione,
età…)
- Ripetizioni / varietà lessicali
- Quali cose importanti
- Errori di scrittura (di tipo sintattico, ortografico)
- Specificità (regionalismi, parole straniere, parole che spiccano come specifiche (molto usate rispetto agli altri profili))
- Paratassi/ ipotassi - complessità sintattica (lunghezza delle frasi) (part of speech - tempi verbali, aggettivi, avverbi)
- Rispetto dei vincoli (coerenza tra testo e profilo)

In [1]:
import pymongo
import pandas as pd 
import numpy as np 
from collections import defaultdict
import matplotlib.pyplot as plt
from nltk.tokenize import word_tokenize, sent_tokenize

In [2]:
db = pymongo.MongoClient()['gentext']['ita']

## Uso di Word2Vec per l'individuazione di *topic*

In [12]:
from gensim.models import Word2Vec
from nltk.tokenize import sent_tokenize, word_tokenize
import numpy as np 
import pandas as pd 

In [6]:
corpus = []
for doc in db.find():
    text = doc['text']
    for sentence in sent_tokenize(text, language='italian'):
        tokens = word_tokenize(sentence.lower(), language='italian')
        corpus.append(tokens)


In [7]:
print(len(corpus))

116756


In [27]:
model = Word2Vec(sentences=corpus, vector_size=300, window=6, 
                        min_count=30, workers=8, epochs=20)

In [32]:
for w, s in model.wv.most_similar('lealtà'):
    print(f"{w} ===> {np.round(s, 4)}")

sincerità ===> 0.6231
l'integrità ===> 0.5922
fedeltà ===> 0.5793
tenacia ===> 0.5503
solidarietà ===> 0.5372
l'onestà ===> 0.5255
precisione ===> 0.5081
compassione ===> 0.506
perseveranza ===> 0.503
carità ===> 0.4778


## Community detection

In [45]:
import networkx as nx 
from string import punctuation
from tqdm.notebook import tqdm

In [58]:
G = nx.Graph()

In [59]:
threshold = 0.65
for word in tqdm(model.wv.index_to_key):
    G.add_node(word)
    if word not in punctuation:
        most_similar = model.wv.most_similar(word)
        for w, s in most_similar:
            if s > threshold:
                G.add_edge(word, w, sim=s)
            else:
                break

  0%|          | 0/1995 [00:00<?, ?it/s]

In [60]:
communities = nx.community.greedy_modularity_communities(G, weight='sim')

In [62]:
for i, c in enumerate(communities):
    print(i, list(c), "\n")

0 ['quaranta', 'proveniente', '14', '24', 'sessantadue', '48', '47', '84', "dell'età", '40', 'trentacinque', 'ventinove', 'nove', '41', 'quarantuno', 'trentasei', 'ottanta', '59', '25', 'ventotto', '29', '78', '36', 'ventiquattro', 'quarantatré', 'settantatré', '50', '16', '76', 'cinquantaquattro', 'quindici', 'venticinque', '55', '43', 'quarantotto', '63', '44', 'avente', '72', '26', '62', '86', 'settantanove', 'cinquanta', "all'età", 'ottantaquattro', '52', '61', '73', 'cinquantanove', 'ventisei', 'quattordici', '35', '28'] 

1 ['ritmi', 'blu', 'paesaggi', 'sorgere', 'boschi', 'montagne', 'alberi', 'silenzio', "all'alba", 'antiche', 'vento', 'cielo', 'campagne', 'luoghi', 'grano', 'segreti', 'vigneti', 'profumo', 'sole', 'vigne', 'verde', 'bosco', 'mare', 'verdi', 'azzurro', 'uccelli', 'colori', 'colline', 'lavorata', 'splendidi', 'monti', "l'aria", 'suono', 'respirare', 'canto', 'parco', 'mattino', 'pura', 'coste', 'animali', 'fiori', 'stagioni', 'valli', 'terre', 'cicli'] 

2 ["del

In [70]:
from sklearn.metrics.pairwise import cosine_similarity

In [74]:
test_c = communities[4]
matrix = []
for word in test_c:
    v = model.wv.get_vector(word)
    matrix.append(v)
M = np.array(matrix)

In [75]:
center = M.mean(axis=0)
sigma = cosine_similarity([center], M)
S = pd.Series(sigma[0], index=list(test_c))

In [77]:
S.sort_values(ascending=False)

chiacchierata    0.885651
caffè            0.881788
amico            0.840071
pasto            0.827986
tramonto         0.814745
sorriso          0.810563
passeggiata      0.774681
abbraccio        0.748089
risata           0.742244
pranzo           0.701759
cena             0.644592
dtype: float32